# Nighthawk Qrack (`nighthawk_qrack.py`) — Google Colab Setup & Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/twobombs/thereminq-examples/blob/main/devices/nighthawk-r3/nighthawk_qrack_colab.ipynb)

This notebook sets up and verifies [`nighthawk_qrack.py`](https://github.com/twobombs/thereminq-examples/blob/main/devices/nighthawk-r3/nighthawk_qrack.py), the consolidated clean-qubit **PyQrack** companion to `arXiv:2609.28657` (random-circuit sampling on IBM Nighthawk r2 / `ibm_phoenix`, 61 qubits) and IBM Nighthawk r3 / multi-chip cloverfield simulations.

### Workflow
1. **Install dependencies** (`pyqrack`, `qiskit`, `ortools`, `scipy`, `matplotlib`, `numpy`, and OpenCL via NVIDIA ICD on GPU runtimes or `pocl-opencl-icd` on CPU runtimes).
2. **Clone repositories** (`twobombs/thereminq-examples` and `BlueQubitDev/rcs-nighthawk`).
3. **Verify circuits & release data** (`verify` and `seamgap`).
4. **Run the self-test suites** (`selftest`, `ghz selftest`, `field selftest`, `erepr selftest`, `lab --selftest`).
5. **Inspect Nighthawk r3 device plans** (`field plan --device r3x2s`, `r3x2`, `clover6`).

## 1. Environment & Dependency Setup

Run the cell below to install Python packages (`pyqrack`, `qiskit`, `ortools`, `scipy`, `matplotlib`) and configure OpenCL (NVIDIA OpenCL ICD when a GPU is detected, or `pocl-opencl-icd` when running on CPU).

In [ ]:
import os
import shutil
import subprocess
import sys

# Install required Python packages
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyqrack", "qiskit", "ortools", "scipy", "matplotlib", "numpy"
])

# Check for GPU via nvidia-smi; if no GPU is detected, install PoCL (pocl-opencl-icd)
has_nvidia = shutil.which("nvidia-smi") is not None and subprocess.call(
    ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
) == 0

subprocess.check_call(["apt-get", "update", "-qq"])
if has_nvidia:
    print("GPU detected via nvidia-smi; configuring NVIDIA OpenCL ICD...")
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ocl-icd-libopencl1", "clinfo"])
    os.makedirs("/etc/OpenCL/vendors", exist_ok=True)
    nv_icd = "/etc/OpenCL/vendors/nvidia.icd"
    if not os.path.exists(nv_icd):
        with open(nv_icd, "w") as f:
            f.write("libnvidia-opencl.so.1\n")
else:
    print("No GPU detected; installing PoCL (pocl-opencl-icd) for CPU OpenCL support...")
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "pocl-opencl-icd", "ocl-icd-libopencl1", "clinfo"])

print("\nOpenCL devices (clinfo -l):")
subprocess.call(["clinfo", "-l"])

import pyqrack
from importlib.metadata import version
print(f"\nPyQrack version: {version('pyqrack')}")
print(f"Qiskit version:  {version('qiskit')}")
print(f"OR-Tools version: {version('ortools')}")

## 2. Clone Repositories & Set Working Directory

This cell clones `twobombs/thereminq-examples` (if not already present), moves into `devices/nighthawk-r3`, and clones the `BlueQubitDev/rcs-nighthawk` dataset next to `nighthawk_qrack.py`.

In [ ]:
from pathlib import Path

if not Path("nighthawk_qrack.py").is_file():
    if not Path("thereminq-examples").is_dir():
        !git clone --depth 1 https://github.com/twobombs/thereminq-examples.git
    %cd thereminq-examples/devices/nighthawk-r3

if not Path("rcs-nighthawk/data/layout.json").is_file():
    !git clone --depth 1 https://github.com/BlueQubitDev/rcs-nighthawk.git rcs-nighthawk

os.environ["NIGHTHAWK_REPO"] = str(Path("rcs-nighthawk").resolve())
print("Working directory:", Path.cwd())
print("NIGHTHAWK_REPO:   ", os.environ["NIGHTHAWK_REPO"])

## 3. Circuit & Release Data Verification (`verify` & `seamgap`)

- `verify` regenerates all 233 circuits from `data/layout.json` and checks them gate-for-gate against `data/circuits/*.qasm`.
- `seamgap` runs the data-only analysis of CZ counts and fidelity vs. depth fits on the released dataset.

In [ ]:
!python3 nighthawk_qrack.py verify --repo rcs-nighthawk
!python3 nighthawk_qrack.py seamgap --repo rcs-nighthawk

## 4. Self-Test Suites

Run the built-in self-tests across all command groups to confirm exactness, angle conventions, Pauli twirling, XEB scoring, CP-SAT layout search, VQE, Shor-9, QrackBridge, GHZ, Cloverfield loom, ER=EPR, and the Configuration Lab.

In [ ]:
# Main self-test suite (14 checks: circuit regeneration, Haar u_angles, exact mirror, Pauli twirl,
# ideal patch XEB, hardware counts XEB, ACE mapping, register recipe, CP-SAT layoutsearch,
# nnqab gate inverses, TFIM VQE, Shor-9, and horizontal QrackBridge)
!python3 nighthawk_qrack.py selftest --repo rcs-nighthawk

In [ ]:
# Subgroup self-tests: ghz, field, erepr, and lab
!python3 nighthawk_qrack.py ghz selftest
!python3 nighthawk_qrack.py field selftest
!python3 nighthawk_qrack.py erepr selftest
!python3 nighthawk_qrack.py lab --selftest

## 5. Nighthawk r3 & Cloverfield Plans and Quick Run

Inspect the multi-chip Nighthawk r3 device presets (`r3x2s`, `r3x2`, `clover6`) and run a quick mirror/Hamming circuit test.

In [ ]:
from IPython.display import Image, display

# Plan the 1,040-qubit two-chip Nighthawk r3 system and render its layout diagram
!python3 nighthawk_qrack.py field plan --device r3x2 --backend exact --png r3x2.png
if Path("r3x2.png").is_file():
    display(Image("r3x2.png"))

# Plan the 240-qubit scaled two-chip r3x2s field on the ghost loom
!python3 nighthawk_qrack.py field plan --device r3x2s --loom ghost

In [ ]:
# List distinct ACE layouts of the 61-qubit register up to max simulator width 28
!python3 nighthawk_qrack.py aceplan --repo rcs-nighthawk --max-width 28